# 1. Gemini API 첫 호출

**목표** — API 키로 Gemini를 호출하고, 돌아온 응답 객체 안에 무엇이 들어있는지 전부 확인한다.

**소요 시간** 약 50분

| 다루는 것 | |
| --- | --- |
| 0 | 환경 확인 |
| 1 | 첫 호출 · 쓸 수 있는 모델 찾기 · 재시도 헬퍼 |
| 2 | 응답 객체 뜯어보기 |
| 3 | `system_instruction` — 역할 부여 |
| 4 | 에러 체험 |
| 5 | 연습문제 |
| 6 | 같은 호출을 OpenAI SDK로 |

> 개념이 헷갈리면 `[배포용] 1_LLM API 동작 원리와 토큰·과금.md`를 먼저 읽는다.

## 0. 환경 확인

`.env`에서 키를 읽어온다. **키 값 자체는 절대 출력하지 않는다** — 노트북에는 실행 결과가 저장되므로, 한 번 찍히면 파일을 공유할 때 딸려나간다.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

# 존재 여부와 길이만 확인 (값은 출력하지 않는다)
key = os.getenv("GEMINI_API_KEY")
print("GEMINI_API_KEY:", "OK" if key else "없음 — [배포용] 0_실습 환경 구성과 API 키 준비.md 4절 참고", f"(길이 {len(key) if key else 0})")

GEMINI_API_KEY: OK (길이 53)


## 1. 첫 호출

`google-genai` SDK로 클라이언트를 만들고 질문을 보낸다. 딱 세 줄이다.

- `genai.Client(api_key=...)` — 어느 계정으로 호출할지 정한다. 한 번만 만들어두고 계속 쓴다.
- `client.models.generate_content(...)` — 실제 호출. 여기서 네트워크 요청이 나간다.
- `response.text` — 생성된 텍스트

모델은 **`gemini-3.1-flash-lite`** 를 쓴다. 응답이 빠르고 무료 한도가 여유로운 편이라 실습 중 `429`가 덜 난다.

> **주의: 모델명은 자주 바뀐다.** 이 셀에서 `404`가 나면 교재가 낡은 것이지 여러분 잘못이 아니다. 바로 아래에서 쓸 수 있는 모델을 찾는 방법을 다룬다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 수도는 어디야? 한 문장으로 답해줘.",
)

print(response.text)
```

In [2]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 응답 텍스트가 출력되면 성공

from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

#MODEL = "gemini-3.1-flash-lite"
MODEL = "gemini-3.5-flash"

response = client.models.generate_content(
    model=MODEL,
    contents="서울 동작구 신대방동 1~2시 사이에 날씨가 어떨지 알려줘 비가올지안올지",
)

print(response.text)

실시간 날씨 정보를 바로 확인하실 수 있도록 가장 빠른 방법을 안내해 드립니다.

제가 실시간 날씨 데이터를 직접 조회하는 기능이 제한되어 있어, 지금 실시간 정보를 바로 보여드리지는 못하지만, 스마트폰으로 **1초 만에 정확하게 확인하시는 방법**은 다음과 같습니다.

1. 포털 사이트(네이버, 다음) 검색창에 **'신대방동 날씨'**라고 검색해 주세요.
2. 화면에 나오는 **'시간별 예보'** 그래프에서 **13시(1시)**와 **14시(2시)** 칸을 확인합니다.
3. **비가 올지 안 올지 판단하는 기준:**
   * **날씨 아이콘:** 우산이나 비구름 모양 그림이 그려져 있는지 확인하세요.
   * **강수확률 (%):** 숫자가 **30% 이상**이면 흐리거나 가끔 빗방울이 떨어질 수 있고, **60% 이상**이면 비가 올 확률이 매우 높으니 외출 시 우산을 꼭 챙기셔야 합니다.

지금 검색창에 **'신대방동 날씨'**를 입력하시면 기상청 레이더와 연동된 가장 정확한 현재 예보를 즉시 확인하실 수 있습니다! 외출 준비에 도움이 되길 바랍니다.


### 여기서 에러가 난다면

| 에러 | 원인 | 내 잘못인가 | 해결 |
| --- | --- | --- | --- |
| `API key not valid` | 키가 틀렸거나 `.env`가 안 읽힘 | 예 | 0번 셀이 `OK`였는지 확인, 키 재발급 |
| `429 RESOURCE_EXHAUSTED` | **내가** 무료 한도 초과 | 예 | 1분 기다렸다 재실행 (노트북 02에서 자세히) |
| `503 UNAVAILABLE` | **구글 서버가** 일시적 과부하 | 아니오 | 잠시 후 재시도. 내 할당량과 무관하다 |
| `404 NOT_FOUND` | 모델명이 없어졌거나 내 프로젝트에 미제공 | 아니오 | 바로 아래 "모델명이 안 먹힐 때" 참고 |
| `ModuleNotFoundError` | 커널이 `.venv`가 아님 | 예 | 우측 상단 커널 선택에서 `.venv` 지정 |

**`429`와 `503`은 성격이 완전히 다르다.** `429`는 내가 너무 많이 불렀다는 뜻이고, `503`은 서버가 붐빈다는 뜻이라 내 할당량은 그대로다. 둘 다 **일시적**이라 재시도로 해결되는데, 이게 실무에서 얼마나 흔한지는 바로 아래에서 다룬다.

### 모델명이 안 먹힐 때 — 실제로 쓸 수 있는 모델 확인하기

**Gemini 모델명은 자주 바뀐다.** 이 교재를 만드는 동안에도 원래 쓰려던 모델이 "신규 사용자에게는 더 이상 제공되지 않는다"며 `404`를 냈다.

더 헷갈리는 점: **`models.list()`에 보이는데도 호출하면 404가 나는 모델이 있다.** 목록에 있다는 건 "그런 모델이 존재한다"는 뜻이지, "내 프로젝트가 쓸 수 있다"는 뜻이 아니다.

그래서 **직접 호출해보는 게 유일하게 확실한 확인 방법**이다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# 후보 모델을 하나씩 실제로 호출해서, 내 키로 쓸 수 있는 것을 찾는다
candidates = [
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash",
    "gemini-3.6-flash",
    "gemini-2.0-flash",
]

for name in candidates:
    try:
        r = client.models.generate_content(model=name, contents="한 단어로: 대한민국 수도")
        u = r.usage_metadata
        print(f"OK    {name:24s} 답={r.text.strip()[:10]!r}  토큰 p/c/t={u.prompt_token_count}/{u.candidates_token_count}/{u.total_token_count}")
    except Exception as e:
        print(f"불가  {name:24s} {str(e)[:60]}")
```

In [3]:
candidates = [
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash",
    "gemini-3.6-flash",
    "gemini-2.0-flash",
]

In [5]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 모델마다 OK / 불가 가 갈리는지 본다
for name in candidates:
    try:
        r = client.models.generate_content(model=name, contents="한 단어로: 대한민국 수도")
        u = r.usage_metadata
        print(f"OK    {name:24s} 답={r.text.strip()[:10]!r}  토큰 p/c/t={u.prompt_token_count}/{u.candidates_token_count}/{u.total_token_count}")
    except Exception as e:
        print(f"불가  {name:24s} {str(e)[:60]}")

OK    gemini-3.1-flash-lite    답='서울'  토큰 p/c/t=7/1/8
불가  gemini-2.5-flash         404 NOT_FOUND. {'error': {'code': 404, 'message': 'This mode
OK    gemini-3.6-flash         답='서울'  토큰 p/c/t=7/1/113
불가  gemini-2.0-flash         404 NOT_FOUND. {'error': {'code': 404, 'message': 'This mode


> **참고: `total`이 `prompt + candidates`보다 훨씬 큰 모델이 있다.** 답변 전에 내부적으로 "생각"하는 추론 모델이라, 그 사고 과정도 토큰으로 과금된다.
> 이 실습에서는 **응답이 빠르고 토큰 계산이 단순한 `gemini-3.1-flash-lite`** 를 쓴다. 위 셀에서 이게 `불가`로 나오면 `OK`인 다른 모델로 `MODEL`을 바꾼다.

### 일시적 오류에 대비하기 — 재시도 헬퍼

`429`와 `503`은 **실습 중에 실제로 자주 만난다.** 특히 아래처럼 반복 호출하는 셀은 한 번만 실패해도 셀 전체가 죽는다.

그래서 앞으로는 이 헬퍼를 통해 호출한다. **잠깐 기다렸다 다시 시도하되, 기다리는 시간을 점점 늘리는 방식(지수 백오프)** 이다. 같은 간격으로 재시도하면 회복되기 전에 또 때려서 상황이 더 나빠진다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
import time


def gen(contents, config=None, retries=4):
    """일시적 오류(429/503)는 기다렸다 재시도한다."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == retries - 1:
                raise                       # 재시도해도 소용없는 오류는 그대로 올린다
            wait = 2 ** attempt             # 1초 → 2초 → 4초
            print(f"  일시적 오류({type(e).__name__}) — {wait}초 후 재시도")
            time.sleep(wait)


print(gen("재시도 헬퍼 테스트. '준비 완료'라고만 답해줘.").text.strip())
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: '준비 완료' 가 출력되면 성공

: 

: 

## 2. 응답 객체 뜯어보기

`response.text`는 편의 속성이고, 실제 응답에는 훨씬 많은 정보가 들어있다.
**특히 `usage_metadata`(토큰 사용량)와 `finish_reason`(왜 멈췄는지)은 앞으로 계속 쓴다.**

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
print("생성된 텍스트:", response.text)
print("실제 사용된 모델:", response.model_version)

candidate = response.candidates[0]
print("종료 이유(finish_reason):", candidate.finish_reason)

usage = response.usage_metadata
print()
print("입력 토큰:", usage.prompt_token_count)
print("출력 토큰:", usage.candidates_token_count)
print("합계  토큰:", usage.total_token_count)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 입력·출력·합계 토큰 세 줄이 나오는지 본다

: 

: 

### 각 값의 의미

| 속성 | 의미 |
| --- | --- |
| `.text` | 생성된 텍스트 (제일 많이 쓴다) |
| `.model_version` | **실제로** 응답한 모델. 요청한 이름과 다를 수 있다 |
| `.candidates[0].finish_reason` | `STOP`=정상 종료 / `MAX_TOKENS`=길이 제한에 잘림 / `SAFETY`=안전 필터 차단 |
| `.usage_metadata.prompt_token_count` | 입력 토큰 수 → **비용의 절반** |
| `.usage_metadata.candidates_token_count` | 출력 토큰 수 → **더 비싼 쪽** |
| `.usage_metadata.total_token_count` | 합계 |

`candidates`가 리스트인 이유는 응답을 여러 개 받을 수도 있기 때문이다. 기본값은 1개라 `[0]`으로 꺼낸다.

## 3. `system_instruction` — 역할 부여

같은 질문이라도 **모델에게 어떤 역할을 주는지**에 따라 답이 완전히 달라진다.
`system_instruction`은 "너는 어떤 존재이고 어떻게 답해야 한다"를 정하는 지침이며, `config`에 담아 보낸다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
question = "블랙홀이 뭐야?"

personas = {
    "지침 없음": None,
    "초등학교 선생님": "너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 2문장 이내로 답한다.",
    "천체물리학자": "너는 천체물리학자다. 전문 용어를 사용해 2문장 이내로 정확하게 설명한다.",
}

for label, instruction in personas.items():
    config = types.GenerateContentConfig(system_instruction=instruction) if instruction else None
    r = gen(question, config)          # 호출을 3번 반복하므로 재시도 헬퍼를 쓴다
    print(f"--- {label} ---")
    print(r.text.strip())
    print()
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 지침에 따라 답변 문체가 달라지는지 본다

: 

: 

같은 `contents`인데 답이 다르다. **모델을 바꾼 게 아니라 지침만 바꿨다.**

> `system_instruction`은 매 호출마다 입력 토큰으로 같이 계산된다. 길게 쓰면 그만큼 매번 비용이 붙는다.

## 4. 에러 체험

실무에서는 **호출이 실패하는 상황을 반드시 처리해야 한다.** 일부러 틀린 요청을 보내서 어떤 에러가 오는지 본다.

`try/except`로 감싸지 않으면 에러 하나에 프로그램 전체가 죽는다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# ① 존재하지 않는 모델명
try:
    client.models.generate_content(model="gemini-없는모델-9.9", contents="안녕")
except Exception as e:
    print("① 잘못된 모델명")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 예외 타입과 404 메시지가 잡히는지 본다

: 

: 

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# ② 잘못된 API 키
bad_client = genai.Client(api_key="AIza-이건-가짜-키-입니다")

try:
    bad_client.models.generate_content(model=MODEL, contents="안녕")
except Exception as e:
    print("② 잘못된 키")
    print("   예외 타입:", type(e).__name__)
    print("   메시지   :", str(e)[:200])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 잘못된 키에서 어떤 예외가 나는지 본다

: 

: 

**에러 메시지를 읽는 습관이 중요하다.** 위 두 에러는 원인이 완전히 다르고, 메시지에 그 단서가 들어있다.
실무에서는 이걸 잡아서 사용자에게 적절한 안내로 바꿔준다(예: `429`면 "잠시 후 다시 시도해주세요").

## 5. 연습문제

### 연습 1-1. 나만의 질문 던지기

아래 TODO를 채워서 본인이 궁금한 것을 물어보고, **응답 텍스트와 총 토큰 수를 함께 출력**한다.

In [ ]:
# TODO: 본인이 궁금한 질문을 넣는다
my_question = ""

# TODO: client.models.generate_content(...) 를 호출해 r 에 담는다
# r = ...

# TODO: 응답 텍스트와 total_token_count 를 출력한다

: 

: 

### 연습 1-2. 페르소나 만들기

`system_instruction`을 직접 작성해서, **같은 질문에 대해 말투가 확 다른 답변 2개**를 만들어본다.
(예: 냉정한 면접관 vs 다정한 멘토)

In [ ]:
# TODO: 서로 대비되는 지침 2개를 작성한다
persona_a = ""
persona_b = ""

my_question = "실패한 프로젝트 경험을 어떻게 말해야 할까요?"

# TODO: 두 지침으로 각각 호출해서 결과를 비교 출력한다

: 

: 

## 6. 같은 호출을 OpenAI SDK로

**SDK는 REST 요청을 감싼 껍데기일 뿐이라, 제공자가 달라도 개념은 그대로다.**
방금 Gemini로 한 것과 똑같은 일을 OpenAI로 해본다.

> **주의:** OpenAI 키는 **강사가 배포한 유료 키**다. 실습 목적으로만 쓰고, 호출을 불필요하게 반복하지 않는다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
from openai import OpenAI

oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

OA_MODEL = "gpt-4o-mini"

oa_response = oa.responses.create(
    model=OA_MODEL,
    instructions="너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 2문장 이내로 답한다.",
    input="블랙홀이 뭐야?",
)

print(oa_response.output_text)
print()
print("입력 토큰:", oa_response.usage.input_tokens)
print("출력 토큰:", oa_response.usage.output_tokens)
print("합계  토큰:", oa_response.usage.total_tokens)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: Gemini와 토큰 수가 다른지 비교한다

: 

: 

### 두 SDK 비교 — 이름만 다르고 개념은 같다

| 개념 | Gemini (`google-genai`) | OpenAI (`openai`) |
| --- | --- | --- |
| 클라이언트 | `genai.Client(api_key=...)` | `OpenAI(api_key=...)` |
| 호출 | `client.models.generate_content(...)` | `client.responses.create(...)` |
| 프롬프트 | `contents=` | `input=` |
| 역할 지침 | `config=GenerateContentConfig(system_instruction=...)` | `instructions=` |
| 결과 텍스트 | `.text` | `.output_text` |
| 입력 토큰 | `.usage_metadata.prompt_token_count` | `.usage.input_tokens` |
| 출력 토큰 | `.usage_metadata.candidates_token_count` | `.usage.output_tokens` |

**표의 왼쪽 열(개념)이 본질이고, 오른쪽 두 열은 표기법 차이다.** 새로운 제공자를 만나도 이 표의 항목만 찾아 매핑하면 된다.

## 정리

- [ ] LLM API를 호출하고 응답 텍스트를 받아봤다
- [ ] `usage_metadata`로 토큰 사용량을 확인했다
- [ ] `finish_reason`이 무엇을 뜻하는지 안다
- [ ] `system_instruction`으로 답변 스타일을 바꿔봤다
- [ ] 호출 실패 시 어떤 에러가 오는지 직접 봤다
- [ ] Gemini와 OpenAI의 코드 구조를 매핑할 수 있다

**다음** → [02_tokens_and_cost.ipynb](./02_tokens_and_cost.ipynb)
방금 본 토큰 숫자가 실제로 얼마인지, 돈으로 계산해본다.